In [5]:
import pathlib
import os
from typing import List

from datasets import load_dataset
import evaluate
import torch
from transformers import (
    GPT2Tokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)

from hf_wrapper import GPTForSequenceClassification
from model import GPT, GPTConfig
from tokenizer import load_tokenizer, eod_token
from sklearn.metrics import precision_score, recall_score


def load_pretrained_model(path: pathlib.Path, device: str = 'cuda') -> GPT:
    checkpoint = torch.load(path, map_location=device)
    gptconf = GPTConfig(**checkpoint['model_args'])
    pretrained_model = GPT(gptconf)
    state_dict = checkpoint['model']

    unwanted_prefix = '_orig_mod.'
    for k, v in list(state_dict.items()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

    model_dict = pretrained_model.state_dict()
    filtered_state_dict = {k: v for k, v in state_dict.items()
                           if k in model_dict and v.shape == model_dict[k].shape}
    model_dict.update(filtered_state_dict)
    pretrained_model.load_state_dict(model_dict)
    pretrained_model.to(device)
    return pretrained_model


# ---- Config ----
args = {
    'train_dataset': 'iggy12345/xnli-en-ipa',
    'eval_dataset': 'iggy12345/xnli-en-ipa',
    'epochs': 3,
    'context_size': 1024,
    'learning_rate': 2e-5,
    'batch_size': 16,
    'hf_cache_dir': pathlib.Path('cache'),
    'device': 'cuda'
}

models_and_tokenizers = [
    {"model_type": "ipa",
     "model_path": "/fs/scratch/PAS2836/ipa_gpt/checkpoints/english_spanish_ipa_12_5_medium_50k_3epoch/ckpt.pt",
     "tokenizer_paths": ("/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-eng-spa-ipa-number-preservation-vocab.json",
                         "/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-eng-spa-ipa-number-preservation-merges.txt")},
    {"model_type": "normal",
     "model_path": "/fs/scratch/PAS2836/ipa_gpt/checkpoints/english_spanish_normal_12_5_medium_50k_3epoch/ckpt.pt",
     "tokenizer_paths": ("/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-eng-spa-normal-number-preservation-vocab.json",
                         "/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-eng-spa-normal-number-preservation-merges.txt")},
]

main_output_dir = pathlib.Path("./training_outputs_es2enmedium")
os.makedirs(main_output_dir, exist_ok=True)

for config in models_and_tokenizers:
    model_type = config["model_type"]
    model_path = config["model_path"]
    tokenizer_paths = config["tokenizer_paths"]

    print(f"\n🔤 Loading {model_type.upper()} model and tokenizer...")

    output_dir = main_output_dir / f"output_{model_type}"
    os.makedirs(output_dir, exist_ok=True)

    # ---- Load Tokenizer ----
    vocab_path, merges_path = tokenizer_paths
    tokenizer = load_tokenizer(vocab_path, merges_path)

    # ---- Load model ----
    base_model = load_pretrained_model(pathlib.Path(model_path), args['device'])
    base_model.config.pad_token_id = tokenizer.pad_token_id
    base_model.config.padding_side = tokenizer.padding_side
    model = GPTForSequenceClassification(base_model, num_classes=3).to(args['device'])

    # ---- Load datasets ----
    train_dataset = load_dataset(args['train_dataset'], split="train", cache_dir=str(args['hf_cache_dir']))
    eval_dataset = load_dataset(args['eval_dataset'], split="validation", cache_dir=str(args['hf_cache_dir']))

    def flatten_multi_features(examples, features: List[str]) -> List[str]:
        separator = f'\n\n{eod_token}\n\n'
        return [separator.join(example) for example in zip(*[examples[f] for f in features])]

    def preprocess_function(examples):
        if model_type == "ipa":
            feature = flatten_multi_features(examples, ['premise-phoneme', 'hypothesis-phoneme'])
        else:
            feature = flatten_multi_features(examples, ['premise', 'hypothesis'])
        return tokenizer(feature, truncation=True, max_length=args['context_size'])

    train_encoded = train_dataset.map(preprocess_function, batched=True)
    eval_encoded = eval_dataset.map(preprocess_function, batched=True)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    metric = evaluate.load("xnli", "en")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = torch.from_numpy(logits).argmax(dim=-1)
        hf_metrics = metric.compute(predictions=predictions, references=labels)
        hf_metrics["precision"] = precision_score(labels, predictions, average="macro")
        hf_metrics["recall"] = recall_score(labels, predictions, average="macro")
        return hf_metrics

    training_args = TrainingArguments(
        output_dir=str(output_dir),
        eval_strategy="steps",
        eval_steps=500,
        save_strategy="steps",
        save_steps=500,
        save_total_limit=1,
        metric_for_best_model="precision",
        load_best_model_at_end=True,
        learning_rate=args['learning_rate'],
        per_device_train_batch_size=args['batch_size'],
        per_device_eval_batch_size=args['batch_size'],
        num_train_epochs=args['epochs'],
        weight_decay=0.01,
        logging_steps=500,
        logging_dir='./logs',
        fp16=True,
        disable_tqdm=False,
        warmup_ratio=0.3,
        save_safetensors=False
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_encoded,
        eval_dataset=eval_encoded,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    print(f"\n🚀 Training {model_type.upper()} model on ES → Evaluating on EN")
    trainer.train(resume_from_checkpoint=False)

    results = trainer.evaluate()
    print(f"\n✅ Evaluation results (ES→EN) for model type: {model_type}:\n{results}")


AttributeError: partially initialized module 'pandas' has no attribute '_pandas_datetime_CAPI' (most likely due to a circular import)

In [6]:
import pandas as pd
print(pd.__version__)


AttributeError: partially initialized module 'pandas' has no attribute '_pandas_datetime_CAPI' (most likely due to a circular import)

In [8]:
!pip install --force-reinstall pandas


Defaulting to user installation because normal site-packages is not writeable
  Using cached numpy-2.0.2-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)

[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
ERROR: Will not install to the user site because it will lack sys.path precedence to python-dateutil in /apps/project/ondemand/app_jupyter/4.1.5/lib/python3.9/site-packages
